# 01 — Data Exploration: ECG200

This notebook performs sanity checks on the ECG200 dataset after preprocessing.
It is meant to be read alongside the thesis draft — every section states what
it checks and why.

**What this notebook covers**
1. Array shapes and split sizes
2. Class balance per split (verifying that stratification worked)
3. Per-class mean trace ± 1 std (are the two classes visually separable?)
4. Raw vs normalised signals side-by-side (does z-scoring look correct?)

**Code organisation note:**  
All data loading and preprocessing logic lives in `src/data/preprocessing.py`.
This notebook contains **no data-processing logic** — it imports from `src/`
and uses the results for exploration only.

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path so `import src` works when running from notebooks/
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.preprocessing import load_ecg200, build_processed_data

# Ensure processed arrays exist (idempotent — safe to call repeatedly)
build_processed_data()

FIGURES_DIR = PROJECT_ROOT / "results" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42

# ── Plot style ────────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", context="notebook", palette="Set2")
plt.rcParams["figure.dpi"] = 120
PALETTE = sns.color_palette("Set2", 2)

print(f"Project root : {PROJECT_ROOT}")
print(f"Figures dir  : {FIGURES_DIR}")

---
## 1. Array shapes

The first thing to confirm is that the preprocessing pipeline produced arrays
with the dimensions we expect:

- ECG200 ships with **100 training samples** and **100 test samples**.
- After an 80 / 20 stratified split from the training portion we expect:
  **80 train / 20 val / 100 test**.
- Each signal has **96 timesteps** (single univariate channel).
- `X` dtype should be `float64`; `y` dtype should be `int`.

In [ ]:
splits = {}
for s in ("train", "val", "test"):
    X, y = load_ecg200(s)
    splits[s] = (X, y)

print(f"{'split':<8} {'X shape':<22} {'y shape':<14} {'X dtype':<12} {'y dtype'}")
print("-" * 70)
for s, (X, y) in splits.items():
    print(f"{s:<8} {str(X.shape):<22} {str(y.shape):<14} {str(X.dtype):<12} {y.dtype}")

print()
n_tr  = splits["train"][0].shape[0]
n_val = splits["val"][0].shape[0]
n_te  = splits["test"][0].shape[0]
print(f"train + val = {n_tr} + {n_val} = {n_tr + n_val}  (expected 100)")
print(f"test        = {n_te}            (expected 100)")
print(f"timesteps   = {splits['train'][0].shape[1]}          (expected  96)")

---
## 2. Class balance

ECG200 is mildly imbalanced: roughly 2/3 of samples are class 1 (normal
heartbeat) and 1/3 are class 0 (abnormal). The stratified split should
**preserve this ratio** across train, val, and test.

What to check:
- All three splits should have a similar `% class 1` value.
- A large discrepancy (> ~5 pp) would indicate a bug in the stratification
  and should be investigated before proceeding to model training.

In [ ]:
print(f"{'split':<8} {'class 0':>10} {'class 1':>10} {'total':>8} {'% class 1':>12}")
print("-" * 54)
for s, (X, y) in splits.items():
    n0 = (y == 0).sum()
    n1 = (y == 1).sum()
    n  = len(y)
    print(f"{s:<8} {n0:>10} {n1:>10} {n:>8} {100 * n1 / n:>11.1f}%")

---
## 3. Per-class mean trace

Plotting the class-conditional mean ± 1 std over the training set gives a
quick visual check that the two classes are distinguishable. If the mean
traces are nearly identical the task would be very hard, and we'd need to
reconsider the preprocessing.

ECG200 class labels (after remapping):
- **Class 0** (raw label −1): abnormal heartbeat
- **Class 1** (raw label  1): normal heartbeat

We plot both classes on shared axes (left) and separately (right) to judge
both inter-class separation and intra-class variance.

In [ ]:
X_train, y_train = splits["train"]
t = np.arange(X_train.shape[1])

CLASS_NAMES = {0: "Class 0 — abnormal", 1: "Class 1 — normal"}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Per-class mean ± 1 std  |  training set (normalised)",
             fontsize=13, y=1.02)

# ── Left: both classes overlaid ───────────────────────────────────────────────
ax_both = axes[0]
for cls in [0, 1]:
    mask   = y_train == cls
    traces = X_train[mask]
    mu     = traces.mean(axis=0)
    sd     = traces.std(axis=0)
    ax_both.plot(t, mu, color=PALETTE[cls], lw=2, label=CLASS_NAMES[cls])
    ax_both.fill_between(t, mu - sd, mu + sd, color=PALETTE[cls], alpha=0.20)
ax_both.set_title("Both classes")
ax_both.set_xlabel("Timestep")
ax_both.set_ylabel("Amplitude (z-scored)")
ax_both.legend(fontsize=9)

# ── Centre / Right: per-class ─────────────────────────────────────────────────
for ax, cls in zip(axes[1:], [0, 1]):
    mask   = y_train == cls
    traces = X_train[mask]
    mu     = traces.mean(axis=0)
    sd     = traces.std(axis=0)
    ax.plot(t, mu, color=PALETTE[cls], lw=2, label=f"Mean (n={mask.sum()})")
    ax.fill_between(t, mu - sd, mu + sd, color=PALETTE[cls], alpha=0.25,
                    label="±1 std")
    ax.set_title(CLASS_NAMES[cls])
    ax.set_xlabel("Timestep")
    ax.set_ylabel("Amplitude (z-scored)")
    ax.legend(fontsize=9)

plt.tight_layout()
out_path = FIGURES_DIR / "01_per_class_means.png"
fig.savefig(out_path, bbox_inches="tight", dpi=150)
plt.show()
print(f"Saved → {out_path}")

---
## 4. Raw vs normalised signals

The preprocessing applies **per-sample z-score normalisation**: each trace is
centred (mean → 0) and scaled (std → 1) using its *own* statistics. This
removes inter-recording baseline drift (e.g. differing electrode offsets across
patients) without leaking any training-set statistics into validation or test
samples.

This section plots two randomly selected examples per class — raw signal on the
left, normalised on the right — so we can visually confirm that:
1. **Shape is preserved**: normalisation should not change the morphology of the
   ECG trace, only its amplitude range.
2. **Amplitude is standardised**: after normalisation, values should be
   approximately in the range [−3, 3] regardless of the original recording scale.
3. **Class differences are still visible**: normalisation should not wash out
   the between-class differences seen in Section 3.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)

# Load raw (un-normalised) training data for comparison
X_tr_raw,  y_tr_raw  = load_ecg200("train", normalised=False)
X_tr_norm, y_tr_norm = splits["train"]   # already loaded above

# Select 2 examples per class — same indices for raw and normalised
selected = []   # list of (class_label, sample_index)
for cls in [0, 1]:
    candidate_idxs = np.where(y_tr_raw == cls)[0]
    chosen = rng.choice(candidate_idxs, size=2, replace=False)
    selected.extend([(cls, int(i)) for i in chosen])

fig, axes = plt.subplots(4, 2, figsize=(13, 11))
fig.suptitle(
    "Raw vs per-sample z-scored normalisation  |  2 examples per class",
    fontsize=13, y=1.01,
)

# Column headers
for ax, col_title in zip(axes[0], ["RAW", "NORMALISED (z-scored)"]):
    ax.annotate(
        col_title, xy=(0.5, 1.18), xycoords="axes fraction",
        ha="center", fontsize=11, fontweight="bold",
        annotation_clip=False,
    )

t = np.arange(X_tr_raw.shape[1])
for row, (cls, idx) in enumerate(selected):
    raw  = X_tr_raw[idx]
    norm = X_tr_norm[idx]
    color = PALETTE[cls]
    label = "abnormal" if cls == 0 else "normal"

    axes[row, 0].plot(t, raw,  color=color, lw=1.2)
    axes[row, 0].set_title(f"Class {cls} ({label})  [sample #{idx}]  — raw",
                            fontsize=9)
    axes[row, 1].plot(t, norm, color=color, lw=1.2, linestyle="--")
    axes[row, 1].set_title(f"Class {cls} ({label})  [sample #{idx}]  — normalised",
                            fontsize=9)

    for ax in axes[row]:
        ax.set_xlabel("Timestep", fontsize=8)
        ax.set_ylabel("Amplitude", fontsize=8)
        ax.tick_params(labelsize=8)

plt.tight_layout()
out_path = FIGURES_DIR / "01_raw_vs_normalised.png"
fig.savefig(out_path, bbox_inches="tight", dpi=150)
plt.show()
print(f"Saved → {out_path}")

---
## Summary

| Check | Expected | Status |
|-------|----------|--------|
| Array shapes | (80, 96) / (20, 96) / (100, 96) | ✓ confirm above |
| Labels in {0, 1} | yes | ✓ by construction |
| Class balance preserved across splits | % class 1 ≈ consistent | ✓ confirm above |
| Classes visually separable | distinct mean traces | ✓ confirm plot |
| Normalisation preserves shape | raw ≈ normalised up to scale | ✓ confirm plot |
| Normalised amplitude range | ≈ [−3, 3] | ✓ confirm plot |

If all checks pass, the dataset is ready for model training.  
Next: `02_model_linear_baseline.ipynb`.